# Step 4: Data Preprocessing & Feature Engineering

This notebook validates the transformation pipeline stored in `src/features/build_features.py` before we blindly apply it to modeling.

In [1]:
import pandas as pd
import sys
import os
import warnings
warnings.filterwarnings('ignore')

# Ensure we can import our src module
sys.path.append(os.path.abspath('..'))
from src.features.build_features import preprocess_data

In [2]:
# To validate the pipeline without waiting 2 minutes for the 6.3m row dataset,
# we'll load a 100,000 row sample to check the output structure.
sample_df = pd.read_csv('../data/raw/Synthetic_Financial_datasets_log.csv', nrows=100000)
print(f"Raw Sample Shape: {sample_df.shape}")

Raw Sample Shape: (100000, 11)


In [3]:
# Execute our preprocessing pipeline
X_scaled, y = preprocess_data(sample_df)

Starting data preprocessing...
Engineering features (error balances)...
Dropping sparse and unique ID columns...
One-hot encoding 'type' column...
Separating features and target...
Scaling numeric columns using RobustScaler...
RobustScaler saved to 'models/robust_scaler.pkl'.


## Validation 1: No Unencoded Strings

In [4]:
X_scaled.info()

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 11 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   amount            100000 non-null  float64
 1   oldbalanceOrg     100000 non-null  float64
 2   newbalanceOrig    100000 non-null  float64
 3   oldbalanceDest    100000 non-null  float64
 4   newbalanceDest    100000 non-null  float64
 5   errorBalanceOrig  100000 non-null  float64
 6   errorBalanceDest  100000 non-null  float64
 7   type_CASH_OUT     100000 non-null  float64
 8   type_DEBIT        100000 non-null  float64
 9   type_PAYMENT      100000 non-null  float64
 10  type_TRANSFER     100000 non-null  float64
dtypes: float64(11)
memory usage: 8.4 MB


## Validation 2: Engineered Features & Outlier Scaling

In [5]:
# Let's look at how the RobustScaler tamed the massive amount numbers 
# (Remember, 0 is the median now)
X_scaled.describe().T

,count,mean,std,min,25%,50%,75%,max
amount,100000.0,0.598895,1.706150,-0.261374,-0.212002,-1.802486e-17,0.787998,49.292739
oldbalanceOrg,100000.0,4.509632,14.055712,-0.105480,-0.105480,0.000000e+00,0.894520,177.595933
newbalanceOrig,100000.0,4.162043,12.621744,0.000000,0.000000,0.000000e+00,1.000000,158.317687
oldbalanceDest,100000.0,1.461339,4.083597,-0.035425,-0.035425,0.000000e+00,0.964575,57.775784
newbalanceDest,100000.0,1.071770,2.648258,-0.047165,-0.047165,0.000000e+00,0.952835,36.757544
errorBalanceOrig,100000.0,0.610522,1.424058,-0.155945,-0.155945,-1.468247e-17,0.844055,25.626335
errorBalanceDest,100000.0,-3.909664,25.493977,-1007.828730,-0.184004,0.000000e+00,0.815996,378.053380
type_CASH_OUT,100000.0,0.307180,0.461327,0.000000,0.000000,0.000000e+00,1.000000,1.000000
type_DEBIT,100000.0,0.009880,0.098906,0.000000,0.000000,0.000000e+00,0.000000,1.000000
type_PAYMENT,100000.0,0.395120,0.488879,0.000000,0.000000,0.000000e+00,1.000000,1.000000


## Validation 3: Imbalance Strategy
As agreed, we are proceeding with **Algorithm-Level Weighting**. We will not perform under/over sampling on the data itself here. During Step 5 (Model Training), we will pass `scale_pos_weight` directly into our XGBoost classifier to force it to mathematically focus on the minority `isFraud=1` cases.